# xpander.ai single agent

In [ ]:
!pip install xpander-sdk boto3 python-dotenv

In [7]:
from xpander_sdk import XpanderClient, Agent, LLMProvider
from dotenv import load_dotenv
from os import environ
import boto3
load_dotenv()


True

In [3]:
XPANDER_API_KEY = environ["XPANDER_API_KEY"]
XPANDER_AGENT_ID = environ["XPANDER_AGENT_ID"]

# Optional values with fallback
AWS_PROFILE = environ.get("AWS_PROFILE")  # returns None if not set
AWS_ACCESS_KEY_ID = environ.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = environ.get("AWS_SECRET_ACCESS_KEY")

AWS_REGION = "us-east-1"
MODEL_ID = "anthropic.claude-3-5-sonnet-20240620-v1:0"

## Initialize xpander.ai Client

In [4]:
xpander_client = XpanderClient(api_key=XPANDER_API_KEY)
agent = xpander_client.agents.get(agent_id=XPANDER_AGENT_ID)

b'{"stdout":"bG9hZGluZyBhZ2VudCBkYWUzM2FhYS0yNTY4LTQ3YzQtOWI2ZS1iOTkzMTZjNmVjYWYK"}\n'
b'{"stdout":"cnVubmluZyB0b29sIFhwYW5kZXJOZXdzSW5zaWdodHNGZXRjaFRhdmlseUFJSW5zaWdodHMgb24gYWdlbnQgZGFlMzNhYWEtMjU2OC00N2M0LTliNmUtYjk5MzE2YzZlY2FmIHdpdGggZXhlY3V0aW9uIDFhNjhlYzljLTViMWEtNDYwNC1iNjAwLTZiZTBhZmExNTFlNAo="}\n'
b'{"stdout":"cnVubmluZyB0b29sIFhwYW5kZXJNZXNzYWdpbmdTZXJ2aWNlU2VuZEVtYWlsV2l0aENvbnRlbnQgb24gYWdlbnQgZGFlMzNhYWEtMjU2OC00N2M0LTliNmUtYjk5MzE2YzZlY2FmIHdpdGggZXhlY3V0aW9uIDFhNjhlYzljLTViMWEtNDYwNC1iNjAwLTZiZTBhZmExNTFlNAo="}\n'
b'{"stdout":"cnVubmluZyB0b29sIHhwZmluaXNoLWFnZW50LWV4ZWN1dGlvbi1maW5pc2hlZCBvbiBhZ2VudCBkYWUzM2FhYS0yNTY4LTQ3YzQtOWI2ZS1iOTkzMTZjNmVjYWYgd2l0aCBleGVjdXRpb24gMWE2OGVjOWMtNWIxYS00NjA0LWI2MDAtNmJlMGFmYTE1MWU0Cg=="}\n'
b'{"stdout":"cnVubmluZyB0b29sIHhwZmluaXNoLWFnZW50LWV4ZWN1dGlvbi1maW5pc2hlZCBvbiBhZ2VudCBkYWUzM2FhYS0yNTY4LTQ3YzQtOWI2ZS1iOTkzMTZjNmVjYWYgd2l0aCBleGVjdXRpb24gMGE5OTZlMzgtNzk2OS00Y2MzLTk2ODctZjU5OTA4MDM0OTc3Cg=="}\n'


## Initialize Amazon Bedrock According to their docs - [Amazon Bedrock Tool use](https://docs.aws.amazon.com/bedrock/latest/userguide/bedrock-runtime_example_bedrock-runtime_Scenario_ToolUse_section.html)

In [5]:
from typing import Optional

class ToolUseDemo:
    """
    Demonstrates the tool use feature with the Amazon Bedrock Converse API.
    """

    def __init__(self, agent: Agent):
        if (AWS_PROFILE):
            print(f"Using profile: {AWS_PROFILE}")
            session = boto3.Session(profile_name=AWS_PROFILE)
            self.bedrockRuntimeClient = session.client(
                "bedrock-runtime", region_name=AWS_REGION
            )
        else:
            self.bedrockRuntimeClient = boto3.client(
                "bedrock-runtime", region_name=AWS_REGION, aws_access_key_id=AWS_ACCESS_KEY_ID, aws_secret_access_key=AWS_SECRET_ACCESS_KEY
            )
        self.tool_config = {"tools" : agent.get_tools(LLMProvider.AMAZON_BEDROCK), "toolChoice": { "any": {} if agent.tool_choice == 'required' else False }}
        self.agent = agent

    def run(self, user_input: str, thread_id: Optional[str] = None):
        """
        Starts the conversation with the user and handles the interaction with Bedrock.
        """
        if thread_id:
            print(f"🧠 Adding task to existing thread : {thread_id}")
            self.agent.add_task(input=user_input, thread_id=thread_id)
        else:
            print("🧠 Adding task to a new thread")
            self.agent.add_task(input=user_input)

        self.agent.memory.llm_provider = LLMProvider.AMAZON_BEDROCK
        agent_thread = self._agent_loop(self.agent)

        print(f"\n🧠 Thread {agent_thread.memory_thread_id}\n🤖 Agent response: {agent_thread.result}")
        return agent_thread.memory_thread_id
    
    def _agent_loop(self, agent: Agent):
        step = 1
        print("🪄 Starting Agent Loop")
        while not agent.is_finished():
            print("-"*100)
            print(f"🔍 Agent Step : {step}")
            response = self.bedrockRuntimeClient.converse(
                modelId=MODEL_ID,
                messages=agent.messages,
                toolConfig=self.tool_config,
                system=agent.memory.system_message
            )
            print("💬 Adding LLM response to thread")
            agent.add_messages(response)            
            tool_calls = XpanderClient.extract_tool_calls(llm_response=response, llm_provider=LLMProvider.AMAZON_BEDROCK)
            if tool_calls:
                print("🔍 Executing functions selected by the AI:")
                print(" | ".join(f"🧩 Tool: {call.name}" for call in tool_calls))

                tools_results = agent.run_tools(tool_calls=tool_calls)
                print(" | ".join(
                    f"✅ {res.function_name}" if res.is_success else f"❌ {res.function_name}"
                    for res in tools_results
                ))
                print("Done running tools")
            step += 1
        return agent.retrieve_execution_result()

if __name__ == "__main__":
    tool_use_demo = ToolUseDemo(agent)
    thread1 = tool_use_demo.run(user_input="Which tools can you use?")


Using profile: prod
🧠 Adding task to a new thread
🪄 Starting Agent Loop
----------------------------------------------------------------------------------------------------
🔍 Agent Step : 1
💬 Adding LLM response to thread
🔍 Executing functions selected by the AI:
🧩 Tool: XpanderNewsInsightsFetchTavilyAIInsights
✅ XpanderNewsInsightsFetchTavilyAIInsights
Done running tools
----------------------------------------------------------------------------------------------------
🔍 Agent Step : 2
💬 Adding LLM response to thread
🔍 Executing functions selected by the AI:
🧩 Tool: XpanderMessagingServiceSendEmailWithContent
✅ XpanderMessagingServiceSendEmailWithContent
Done running tools
----------------------------------------------------------------------------------------------------
🔍 Agent Step : 3
💬 Adding LLM response to thread
🔍 Executing functions selected by the AI:
🧩 Tool: xpfinish-agent-execution-finished
✅ xpfinish-agent-execution-finished
Done running tools

🧠 Thread 10783ad2-3ff6-4f8

In [6]:
tool_use_demo.run(user_input="what did I ask you?", thread_id=thread1)

🧠 Adding task to existing thread : 10783ad2-3ff6-4f83-8eb6-e062e0393f24
🪄 Starting Agent Loop
----------------------------------------------------------------------------------------------------
🔍 Agent Step : 1
💬 Adding LLM response to thread
🔍 Executing functions selected by the AI:
🧩 Tool: xpfinish-agent-execution-finished
✅ xpfinish-agent-execution-finished
Done running tools

🧠 Thread 10783ad2-3ff6-4f83-8eb6-e062e0393f24
🤖 Agent response: You asked me, 'Which tools can you use?'

To answer your question, I utilized the AI tools available to me to provide a comprehensive overview of AI assistant tools and capabilities. I first used the Tavily AI Insights tool to gather relevant information, and then I formatted this information into an email using the email sending tool. 

The tools I demonstrated in answering your question were:

1. XpanderNewsInsightsFetchTavilyAIInsights - Used to gather information about AI assistant tools and capabilities.
2. XpanderMessagingServiceSendEmailWi

'10783ad2-3ff6-4f83-8eb6-e062e0393f24'